# 27 — Skills Catalog And Usage

Use this notebook to fetch the packaged SHIPIT skills catalog and apply prebuilt skills with both `Agent` and `DeepAgent`.

In [ ]:
from pathlib import Path
import sys

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display
from examples.run_multi_tool_agent import build_llm_from_env
from shipit_agent import Agent, DeepAgent, DEFAULT_SKILLS_PATH, FileSkillRegistry

llm = build_llm_from_env("bedrock")
registry = FileSkillRegistry(DEFAULT_SKILLS_PATH)

## 1. Fetch All Skills

The packaged catalog is stored in `shipit_agent/skills/skills.json`. This cell loads every skill and shows a compact table.

In [ ]:
all_skills = registry.list()
print(f"Catalog path: {DEFAULT_SKILLS_PATH}")
print(f"Total skills: {len(all_skills)}")

for skill in all_skills[:15]:
    print(f"- {skill.id:30} | {skill.category or 'uncategorized':18} | {skill.display_name or skill.name}")

## 2. Search The Catalog

Use fuzzy search to find relevant prebuilt skills before attaching them to an agent.

In [ ]:
query = "database"
matches = registry.search(query)
print(f"Search query: {query!r}")
for skill in matches[:10]:
    print(f"- {skill.id}: {skill.description}")

## 3. Build An Agent With Prebuilt Skills

You can pass skill ids directly with `skills=[...]`. These skills are always attached. Auto-matching can stay enabled or be disabled.

In [ ]:
agent = Agent.with_builtins(
    llm=llm,
    name="skills-demo-agent",
    skills=["database-architect", "code-workflow-assistant"],
    default_skill_ids=["brand-voice-guide"],
    auto_use_skills=True,
    skill_match_limit=2,
)

print("Attached skills:")
for skill in agent.skills:
    print(f"- {skill.id}")

print("\nSearch from the live agent:")
for skill in agent.search_skills("workflow")[:5]:
    print(f"- {skill.id}")

## 4. Use The Agent

This run combines explicit attached skills with automatic skill matching from the prompt.

In [ ]:
result = agent.run(
    "Plan this feature, debug the failing query path, and suggest database improvements."
)

display(Markdown(result.output))

## 5. Add A Skill At Runtime

You can attach another prebuilt skill after the agent is created.

In [ ]:
added = agent.add_skill("portfolio-website-builder")
print(f"Added skill: {added.id}")
print("All attached skills now:")
for skill in agent.skills:
    print(f"- {skill.id}")

## 6. Build A DeepAgent With Skills

`DeepAgent` supports the same skill API and passes it through to the inner `Agent`.

In [ ]:
deep_agent = DeepAgent.with_builtins(
    llm=llm,
    name="skills-demo-deep-agent",
    skills=["database-architect"],
    default_skill_ids=["code-workflow-assistant"],
    auto_use_skills=True,
)

print("DeepAgent attached skills:")
for skill in deep_agent.skills:
    print(f"- {skill.id}")

print("\nDeepAgent catalog search:")
for skill in deep_agent.search_skills("brand")[:5]:
    print(f"- {skill.id}")

In [ ]:
deep_result = deep_agent.run(
    "Investigate a slow database-backed feature, make a plan, and review the implementation approach."
)

display(Markdown(getattr(deep_result, "output", str(deep_result))))

## 7. Minimal Patterns

Common patterns:

- `skills=[...]`: always attach these prebuilt skills.
- `default_skill_ids=[...]`: attach by id from the packaged catalog.
- `auto_use_skills=True`: also match skills from the user prompt.
- `skill_match_limit=...`: cap the number of auto-matched skills.
- `agent.add_skill("...")`: attach another skill later.